# Embeddings & Vector Representations Notebook

> Hands-on Build It and Exercises.

## Build It

We build a semantic search engine from scratch. No vector database. No external embedding API. Pure Python with numpy for the math.

### Step 1: Text Chunking

In [ ]:
```python

def chunk_text(text, chunk_size=200, overlap=50):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

def chunk_by_sentences(text, max_chunk_tokens=200):

    sentences = text.replace("\n", " ").split(".")

    sentences = [s.strip() + "." for s in sentences if s.strip()]

    chunks = []

    current_chunk = []

    current_length = 0

    for sentence in sentences:

        sentence_length = len(sentence.split())

        if current_length + sentence_length > max_chunk_tokens and current_chunk:

            chunks.append(" ".join(current_chunk))

            current_chunk = []

            current_length = 0

        current_chunk.append(sentence)

        current_length += sentence_length

    if current_chunk:

        chunks.append(" ".join(current_chunk))

    return chunks

In [ ]:
```

### Step 2: Building Embeddings from Scratch

We implement a simple dense embedding using TF-IDF with L2 normalization. This is not a neural embedding, but it follows the same contract: text in, fixed-size vector out, similar texts produce similar vectors.

In [ ]:
```python

import math

import numpy as np

from collections import Counter

class SimpleEmbedder:

    def __init__(self):

        self.vocab = []

        self.idf = []

        self.word_to_idx = {}

    def fit(self, documents):

        vocab_set = set()

        for doc in documents:

            vocab_set.update(doc.lower().split())

        self.vocab = sorted(vocab_set)

        self.word_to_idx = {w: i for i, w in enumerate(self.vocab)}

        n = len(documents)

        self.idf = np.zeros(len(self.vocab))

        for i, word in enumerate(self.vocab):

            doc_count = sum(1 for doc in documents if word in doc.lower().split())

            self.idf[i] = math.log((n + 1) / (doc_count + 1)) + 1

    def embed(self, text):

        words = text.lower().split()

        count = Counter(words)

        total = len(words) if words else 1

        vec = np.zeros(len(self.vocab))

        for word, freq in count.items():

            if word in self.word_to_idx:

                tf = freq / total

                vec[self.word_to_idx[word]] = tf * self.idf[self.word_to_idx[word]]

        norm = np.linalg.norm(vec)

        if norm > 0:

            vec = vec / norm

        return vec

In [ ]:
```

### Step 3: Similarity Functions

In [ ]:
```python

def cosine_similarity(a, b):

    dot = np.dot(a, b)

    norm_a = np.linalg.norm(a)

    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:

        return 0.0

    return float(dot / (norm_a * norm_b))

def dot_product(a, b):

    return float(np.dot(a, b))

def euclidean_distance(a, b):

    return float(np.linalg.norm(a - b))

In [ ]:
```

### Step 4: Vector Index with Brute-Force Search

In [ ]:
```python

class VectorIndex:

    def __init__(self):

        self.vectors = []

        self.texts = []

        self.metadata = []

    def add(self, vector, text, meta=None):

        self.vectors.append(vector)

        self.texts.append(text)

        self.metadata.append(meta or {})

    def search(self, query_vector, top_k=5, metric="cosine"):

        scores = []

        for i, vec in enumerate(self.vectors):

            if metric == "cosine":

                score = cosine_similarity(query_vector, vec)

            elif metric == "dot":

                score = dot_product(query_vector, vec)

            elif metric == "euclidean":

                score = -euclidean_distance(query_vector, vec)

            else:

                raise ValueError(f"Unknown metric: {metric}")

            scores.append((i, score))

        scores.sort(key=lambda x: x[1], reverse=True)

        results = []

        for idx, score in scores[:top_k]:

            results.append({

                "text": self.texts[idx],

                "score": score,

                "metadata": self.metadata[idx],

                "index": idx

            })

        return results

    def size(self):

        return len(self.vectors)

In [ ]:
```

### Step 5: The Semantic Search Engine

In [ ]:
```python

class SemanticSearchEngine:

    def __init__(self, chunk_size=200, overlap=50):

        self.embedder = SimpleEmbedder()

        self.index = VectorIndex()

        self.chunk_size = chunk_size

        self.overlap = overlap

    def index_documents(self, documents, source_names=None):

        all_chunks = []

        all_sources = []

        for i, doc in enumerate(documents):

            chunks = chunk_text(doc, self.chunk_size, self.overlap)

            all_chunks.extend(chunks)

            name = source_names[i] if source_names else f"doc_{i}"

            all_sources.extend([name] * len(chunks))

        self.embedder.fit(all_chunks)

        for chunk, source in zip(all_chunks, all_sources):

            vec = self.embedder.embed(chunk)

            self.index.add(vec, chunk, {"source": source})

        return len(all_chunks)

    def search(self, query, top_k=5, metric="cosine"):

        query_vec = self.embedder.embed(query)

        return self.index.search(query_vec, top_k, metric)

    def search_with_scores(self, query, top_k=5):

        results = self.search(query, top_k)

        return [

            {

                "text": r["text"][:200],

                "source": r["metadata"].get("source", "unknown"),

                "score": round(r["score"], 4)

            }

            for r in results

        ]

In [ ]:
```

### Step 6: Comparing Similarity Metrics

In [ ]:
```python

def compare_metrics(engine, query, top_k=3):

    results = {}

    for metric in ["cosine", "dot", "euclidean"]:

        hits = engine.search(query, top_k=top_k, metric=metric)

        results[metric] = [

            {"score": round(h["score"], 4), "preview": h["text"][:80]}

            for h in hits

        ]

    return results

In [ ]:
```

## Exercises

In [ ]:
1. **Metric comparison**: run the same 5 queries against the sample documents using cosine similarity, dot product, and euclidean distance. Record the top-3 results for each. For which queries do the metrics disagree? Why?

2. **Chunk size experiment**: index the sample documents with chunk sizes of 50, 100, 200, and 500 words. For each, run 5 queries and record the top-1 similarity score. Plot the relationship between chunk size and retrieval quality. Find the point where larger chunks start hurting.

3. **Why truncation needs training**: build a SimpleEmbedder that produces 500-d TF-IDF vectors. Compare three truncation strategies at 50, 100, 200, and 500 dimensions: (a) keep the first N dimensions as-is (alphabetically-first vocabulary words, per `fit()`), (b) keep the N dimensions with the highest average IDF weight across your corpus, (c) keep N random dimensions. Measure retrieval recall for each strategy at each size. Predict before you run it: which strategy degrades most gracefully? Strategy (b) should beat (a) and (c) -- but even (b) won't match a real Matryoshka model's degradation curve, because nothing here was trained with the nested-loss objective from "Matryoshka Embeddings" above. That gap is the point: reordering dimensions by importance *after* training helps a little; Matryoshka's actual trick is training the model to put importance there in the first place.

4. **Binary quantization**: take the embeddings from the search engine, convert them to binary (1 if positive, 0 if negative), and implement Hamming distance search. Compare the top-10 results against full-precision cosine similarity. Measure the overlap percentage.

5. **Sentence-based chunking**: replace fixed-size chunking with `chunk_by_sentences`. Run the same queries and compare retrieval scores. Does respecting sentence boundaries improve the results?